# 🩺 MedTranslate: Patient-First Medical AI Explainer
### Powered by Google Health **MedGemma** (`google/medgemma-4b-it`)

This notebook demonstrates how to load Google's open-weights **MedGemma 4B** multimodal model on a **Free Google Colab T4 GPU**, run inference on medical images (X-rays, Dermoscopy) and clinical notes, and launch an interactive **Gradio Web App** with a shareable public link.

> ⚠️ **Disclaimer:** For educational, research demonstration, and health literacy exploration only. Not approved for clinical diagnosis or treatment.

## 1. Check GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q --upgrade transformers>=4.50.0 torch accelerate gradio pillow huggingface_hub

## 2. Hugging Face Login (MedGemma Access)
MedGemma models are open-weights under Google's HAI-DEF license. Make sure you accept the license agreement on the [google/medgemma-4b-it](https://huggingface.co/google/medgemma-4b-it) model card and enter your HF token below.

In [ ]:
from huggingface_hub import login
# Enter your Hugging Face User Access Token (read permission)
login()

## 3. Load MedGemma 4B Multimodal Model

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image

model_id = "google/medgemma-4b-it"
print(f"Loading {model_id} onto GPU...")

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("Model loaded successfully!")

## 4. Test Single Multimodal Inference (Sample Chest X-Ray)

In [ ]:
# Download a sample open-source chest X-ray
!wget -q -O sample_cxr.png "https://raw.githubusercontent.com/Google-Health/medgemma/main/examples/data/sample_chest_xray.png" || true

import os
if not os.path.exists("sample_cxr.png"):
    from PIL import Image, ImageDraw
    img = Image.new("RGB", (256, 256), color=(73, 109, 137))
    d = ImageDraw.Draw(img)
    d.text((20, 120), "Sample Medical Scan", fill=(255, 255, 0))
    img.save("sample_cxr.png")

sample_image = Image.open("sample_cxr.png")

prompt = (
    "You are MedGemma, a clinical AI assistant. Review this radiograph. "
    "Provide:\n"
    "1. Clinical Observations\n"
    "2. A patient-friendly explanation in plain English\n"
    "3. Three clarifying questions the patient should ask their doctor."
)

inputs = processor(text=prompt, images=sample_image, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=450)

response_text = processor.decode(outputs[0], skip_special_tokens=True)
print(response_text)

## 5. Launch Full Gradio Web App with Public Shareable Link
Running the cell below launches the web app and gives you a **`gradio.live`** public link you can open on any device.

In [ ]:
import gradio as gr

def run_medical_image_analysis(img, clinical_notes):
    if img is None:
        return "Please provide an image."
    prompt = (
        "You are MedGemma. Analyze this medical image and clinical notes.\n"
        f"Notes: {clinical_notes if clinical_notes else 'None'}\n"
        "Structure your response:\n"
        "### 🩺 Clinical Findings (Clinician Draft)\n"
        "### 💡 Plain-English Summary (Patient Guide)\n"
        "### ❓ Questions to Ask Your Doctor\n"
    )
    inputs = processor(text=prompt, images=img, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_tokens = model.generate(**inputs, max_new_tokens=500)
    return processor.decode(output_tokens[0], skip_special_tokens=True)

def run_report_translation(report_text):
    if not report_text:
        return "Please enter medical report text."
    prompt = (
        "You are MedGemma. Translate this clinical report into plain English for a patient. "
        "Include a glossary of any medical terms and 3 questions they should ask their physician.\n\n"
        f"Report:\n{report_text}"
    )
    inputs = processor(text=prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_tokens = model.generate(**inputs, max_new_tokens=500)
    return processor.decode(output_tokens[0], skip_special_tokens=True)

with gr.Blocks(title="MedTranslate Demo", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🩺 MedTranslate (Powered by MedGemma 4B)")
    gr.Markdown("**Disclaimer:** For research demonstration and patient health literacy exploration only.")
    
    with gr.Tab("🖼️ Multimodal Medical Imaging"):
        with gr.Row():
            with gr.Column():
                img_in = gr.Image(type="pil", label="Chest X-Ray / Medical Scan")
                note_in = gr.Textbox(label="Clinical Notes / Symptoms", placeholder="Patient reports 3-day cough and mild fever...")
                img_btn = gr.Button("Analyze Scan", variant="primary")
            with gr.Column():
                img_out = gr.Markdown(label="MedGemma Output")
        img_btn.click(fn=run_medical_image_analysis, inputs=[img_in, note_in], outputs=img_out)
        
    with gr.Tab("📄 Clinical Report & Lab Translator"):
        with gr.Row():
            with gr.Column():
                txt_in = gr.Textbox(lines=6, label="Medical Report / Lab Values", placeholder="Paste report text or lab results here...")
                txt_btn = gr.Button("Translate to Plain English", variant="primary")
            with gr.Column():
                txt_out = gr.Markdown(label="MedGemma Output")
        txt_btn.click(fn=run_report_translation, inputs=txt_in, outputs=txt_out)

demo.launch(share=True, debug=True)